# MCP Server Deployment in Bedrock AgentCore Runtime

This workshop demonstrates how to deploy and use Model Context Protocol (MCP) servers with Amazon Bedrock AgentCore Runtime, enabling scalable and secure deployment of custom tools for AI agents.

## Overview

In this lab, you will:
- Create a custom MCP server with web search functionality
- Set up authentication using Amazon Cognito
- Deploy the MCP server to Bedrock AgentCore Runtime
- Test the deployed server with Strands Agents

## What is Bedrock AgentCore Runtime for MCP?

Amazon Bedrock AgentCore Runtime allows you to deploy Model Context Protocol (MCP) servers as managed, scalable services. Key benefits include:

- **Scalability**: Automatically scales based on demand
- **Security**: Built-in authentication and authorization
- **Managed Infrastructure**: No need to manage servers or containers
- **Integration**: Seamless integration with Bedrock services

MCP servers provide tools and resources that AI agents can use to extend their capabilities, such as web search, database access, or custom business logic.

## Prerequisites

In [ ]:
import os
#os.environ['AWS_ACCESS_KEY_ID'] = ''
#os.environ['AWS_SECRET_ACCESS_KEY'] = ''
#os.environ['AWS_SESSION_TOKEN'] = ''
#os.environ['AWS_REGION'] = ''

In [ ]:
#%pip install -q strands-agents strands-agents-tools bedrock-agentcore mcp fastmcp ddgs rich

In [1]:
import boto3
region = boto3.session.Session().region_name
NOVA_PRO_MODEL_ID = 'us.amazon.nova-pro-v1:0'
if region.startswith('eu'): NOVA_PRO_MODEL_ID = 'eu.amazon.nova-pro-v1:0'
elif region.startswith('ap'): NOVA_PRO_MODEL_ID = 'apac.amazon.nova-pro-v1:0'
print(f'Region: {region}, Model: {NOVA_PRO_MODEL_ID}')

Region: ap-southeast-2, Model: apac.amazon.nova-pro-v1:0


### Creating a Custom MCP Server

Let's create a simple MCP server that provides web search functionality using DuckDuckGo. This server will be deployed to AgentCore Runtime for scalable use.

In [3]:
%%writefile mcp_server.py
from mcp.server.fastmcp import FastMCP
from ddgs import DDGS
from ddgs.exceptions import RatelimitException, DDGSException

mcp = FastMCP(host="0.0.0.0", stateless_http=True)


@mcp.tool()
def websearch(keywords: str, region: str = "us-en", max_results: int | None = None) -> list:
    """Search the web to get updated information.
    Args:
        keywords (str): The search query keywords.
        region (str): The search region: wt-wt, us-en, uk-en, ru-ru, etc..
        max_results (int | None): The maximum number of results to return.
    Returns:
        List of dictionaries with search results.
    """
    try:
        results = DDGS().text(keywords, region=region, max_results=max_results)
        return results if results else "No results found."
    except RatelimitException:
        return "RatelimitException: Please try again after a short delay."
    except DDGSException as d:
        return f"DuckDuckGoSearchException: {d}"
    except Exception as e:
        return f"Exception: {e}"


@mcp.tool()
def validate_bsb(bsb: str) -> dict:
    """Validate an Australian BSB (Bank-State-Branch) number and identify the bank.
    Args:
        bsb (str): The BSB number to validate (e.g., "062-000" or "062000").
    Returns:
        Dictionary with validation result, bank name, and details.
    """
    bsb_bank_map = {
        "01": "ANZ Banking Group", "03": "Westpac Banking Corporation",
        "06": "Commonwealth Bank of Australia", "08": "National Australia Bank",
        "09": "Reserve Bank of Australia", "12": "Bank of Queensland",
        "18": "Macquarie Bank", "30": "Bankwest", "33": "St George Bank",
        "34": "HSBC Bank Australia", "42": "Deutsche Bank", "48": "Rabobank",
        "55": "Bank of China", "73": "Westpac (BankSA/St George)",
        "76": "Commonwealth Bank (Bankwest)", "92": "Westpac (BankSA)",
    }
    clean = bsb.replace("-", "").replace(" ", "")
    if len(clean) != 6 or not clean.isdigit():
        return {"valid": False, "error": "BSB must be 6 digits"}
    prefix = clean[:2]
    bank = bsb_bank_map.get(prefix)
    formatted = f"{clean[:3]}-{clean[3:]}"
    if bank:
        return {"valid": True, "bsb": formatted, "bank": bank}
    return {"valid": False, "bsb": formatted, "error": f"Unknown bank prefix: {prefix}"}


if __name__ == "__main__":
    mcp.run(transport="streamable-http")


Overwriting mcp_server.py


### Local Testing of MCP Server (Optional)

Before deploying to AgentCore Runtime, test the MCP server locally.

**Start the MCP Server in a separate terminal:**
```bash
cd 04-agentcore-runtime-mcp/
pip install mcp fastmcp ddgs
python mcp_server.py
```

Then run the cell below to test:

In [ ]:
from strands import Agent
from strands.models import BedrockModel
from strands.tools.mcp import MCPClient
from mcp.client.streamable_http import streamablehttp_client

try:
    mcp_url = 'http://localhost:8000/mcp'
    mcp_client = MCPClient(lambda: streamablehttp_client(mcp_url))

    with mcp_client:
        tools = mcp_client.list_tools_sync()
        print(f'Available tools: {[t.tool_name for t in tools]}')

        agent = Agent(
            model=BedrockModel(model_id=NOVA_PRO_MODEL_ID, max_tokens=4096),
            system_prompt='You are a financial research assistant. Use web search to find current information.',
            tools=tools,
        )
        agent('Validate BSB 062-000 and also search to confirm if ABN 77003236628 belongs to Amazon')
except Exception as e:
    print(f'Local server not running (expected if skipping): {e}')

### Stop the MCP server running locally

After testing locally, press `Ctrl+C` in the terminal to stop the MCP server.

## Deploy MCP Server in Bedrock AgentCore Runtime with Authentication

Now we'll configure and deploy our MCP server to Bedrock AgentCore Runtime.

### Step 1: Setting up Amazon Cognito for Inbound Authentication

Create a Cognito User Pool for secure access to the deployed MCP server.

In [4]:
import boto3

cognito_client = boto3.client('cognito-idp', region_name=region)

# Create User Pool
pool_name = 'fsi_mcp_auth_pool'
try:
    pool_response = cognito_client.create_user_pool(
        PoolName=pool_name,
        Policies={'PasswordPolicy': {'MinimumLength': 8}},
        AutoVerifiedAttributes=['email'],
    )
    user_pool_id = pool_response['UserPool']['Id']
    print(f'✅ User Pool created: {user_pool_id}')
except Exception as e:
    if 'already exists' in str(e).lower():
        pools = cognito_client.list_user_pools(MaxResults=50)['UserPools']
        user_pool_id = next(p['Id'] for p in pools if p['Name'] == pool_name)
        print(f'✅ Using existing pool: {user_pool_id}')
    else:
        raise e

# Create App Client
client_response = cognito_client.create_user_pool_client(
    UserPoolId=user_pool_id,
    ClientName='fsi_mcp_client',
    ExplicitAuthFlows=['ALLOW_USER_PASSWORD_AUTH', 'ALLOW_REFRESH_TOKEN_AUTH'],
    GenerateSecret=False,
)
client_id = client_response['UserPoolClient']['ClientId']
print(f'✅ App Client created: {client_id}')

# Create test user
test_password = 'FsiDemo2026!'
try:
    cognito_client.admin_create_user(
        UserPoolId=user_pool_id,
        Username='fsi_agent',
        TemporaryPassword=test_password,
        MessageAction='SUPPRESS',
    )
    cognito_client.admin_set_user_password(
        UserPoolId=user_pool_id,
        Username='fsi_agent',
        Password=test_password,
        Permanent=True,
    )
    print(f'✅ Test user created: fsi_agent')
except cognito_client.exceptions.UsernameExistsException:
    print(f'✅ Test user already exists: fsi_agent')

✅ User Pool created: ap-southeast-2_pWcVvNxRi
✅ App Client created: 4i212f2pdmub1cad5ei2uf6aav
✅ Test user created: fsi_agent


### Step 2: Configuring and Deploying to Bedrock AgentCore Runtime

Set up and deploy the MCP server to AgentCore Runtime with automatic resource creation.

In [6]:
from bedrock_agentcore_starter_toolkit import Runtime

agentcore_runtime = Runtime()

print('Configuring AgentCore Runtime...')
response = agentcore_runtime.configure(
    entrypoint='mcp_server.py',
    auto_create_execution_role=True,
    auto_create_ecr=True,
    requirements_file='requirements.txt',
    region=region,
    protocol='MCP',
    agent_name='fsi_websearch_mcp',
    non_interactive=True,
    authorizer_configuration={
        'customJWTAuthorizer': {
            'allowedClients': [client_id],
            'discoveryUrl': f'https://cognito-idp.{region}.amazonaws.com/{user_pool_id}/.well-known/openid-configuration',
        }
    }
)
print('Configuration completed ✓')

print('\nLaunching deployment (3-5 minutes)...')
launch_result = agentcore_runtime.launch()
mcp_runtime_arn = launch_result.agent_arn
print(f'\n✅ Deployed! ARN: {mcp_runtime_arn}')

Entrypoint parsed: file=/Users/zohaibso/AI Workshops/FSI-AgentCore-Workshop/04-agentcore-runtime-mcp/mcp_server.py, bedrock_agentcore_name=mcp_server
Memory disabled - agent will be stateless
Configuring BedrockAgentCore agent: fsi_websearch_mcp


Configuring AgentCore Runtime...


💡 No container engine found (Docker/Finch/Podman not installed)

✓ Default deployment uses CodeBuild (no container engine needed), For local builds, install Docker, Finch, or 
Podman

Memory disabled
Network mode: PUBLIC


📄 Using existing Dockerfile: /Users/zohaibso/AI 
Workshops/FSI-AgentCore-Workshop/04-agentcore-runtime-mcp/Dockerfile

Generated .dockerignore: /Users/zohaibso/AI Workshops/FSI-AgentCore-Workshop/04-agentcore-runtime-mcp/.dockerignore
Keeping 'fsi_websearch_mcp' as default agent
Bedrock AgentCore configured: /Users/zohaibso/AI Workshops/FSI-AgentCore-Workshop/04-agentcore-runtime-mcp/.bedrock_agentcore.yaml
🚀 Launching Bedrock AgentCore (cloud mode - RECOMMENDED)...
   • Deploy Python code directly to runtime
   • No Docker required (DEFAULT behavior)
   • Production-ready deployment

💡 Deployment options:
   • runtime.launch()                → Cloud (current)
   • runtime.launch(local=True)      → Local development
Memory disabled - skipping memory creation
Starting CodeBuild ARM64 deployment for agent 'fsi_websearch_mcp' to account 905035168378 (ap-southeast-2)
Generated image tag: 20260605-044014-123
Setting up AWS resources (ECR repository, execution roles)...
Getting or creating ECR repository for agent: fsi_websearch_mcp
ECR repository available: 905035168378.dkr.ecr.ap-southeast-2.amazonaws.com/

Configuration completed ✓

Launching deployment (3-5 minutes)...
✅ Reusing existing ECR repository: 905035168378.dkr.ecr.ap-southeast-2.amazonaws.com/bedrock-agentcore-fsi_websearch_mcp


✅ Reusing existing execution role: arn:aws:iam::905035168378:role/AmazonBedrockAgentCoreSDKRuntime-ap-southeast-2-f4993fa5e3
Execution role available: arn:aws:iam::905035168378:role/AmazonBedrockAgentCoreSDKRuntime-ap-southeast-2-f4993fa5e3
Preparing CodeBuild project and uploading source...
Getting or creating CodeBuild execution role for agent: fsi_websearch_mcp
Role name: AmazonBedrockAgentCoreSDKCodeBuild-ap-southeast-2-f4993fa5e3
Reusing existing CodeBuild execution role: arn:aws:iam::905035168378:role/AmazonBedrockAgentCoreSDKCodeBuild-ap-southeast-2-f4993fa5e3
Using dockerignore.template with 47 patterns for zip filtering
Uploaded source to S3: fsi_websearch_mcp/source.zip
Updated CodeBuild project: bedrock-agentcore-fsi_websearch_mcp-builder
Starting CodeBuild build (this may take several minutes)...
Starting CodeBuild monitoring...
🔄 QUEUED started (total: 0s)
✅ QUEUED completed in 1.1s
🔄 PROVISIONING started (total: 1s)
✅ PROVISIONING completed in 8.6s
🔄 DOWNLOAD_SOURCE start


✅ Deployed! ARN: arn:aws:bedrock-agentcore:ap-southeast-2:905035168378:runtime/fsi_websearch_mcp-Odq9bg3tKv


### Testing the Deployed MCP Server with Strands Agent

Now let's test our deployed MCP server by connecting through the AgentCore Runtime endpoint with Cognito authentication.

In [7]:
# Get Cognito access token
auth_response = cognito_client.initiate_auth(
    ClientId=client_id,
    AuthFlow='USER_PASSWORD_AUTH',
    AuthParameters={'USERNAME': 'fsi_agent', 'PASSWORD': test_password},
)
access_token = auth_response['AuthenticationResult']['AccessToken']
print(f'✅ Authenticated. Token: {access_token[:20]}...')

✅ Authenticated. Token: eyJraWQiOiI0KzY0a2Jx...


In [8]:
from strands import Agent
from strands.models import BedrockModel
from strands.tools.mcp import MCPClient
from mcp.client.streamable_http import streamablehttp_client
import warnings
warnings.filterwarnings("ignore")

encoded_arn = mcp_runtime_arn.replace(":", "%3A").replace("/", "%2F")
mcp_url = f"https://bedrock-agentcore.{region}.amazonaws.com/runtimes/{encoded_arn}/invocations"
headers = {"Authorization": f"Bearer {access_token}"}

print(f"Connecting to deployed MCP server...")
mcp_client = MCPClient(lambda: streamablehttp_client(mcp_url, headers=headers))

try:
    with mcp_client:
        tools = mcp_client.list_tools_sync()
        print(f"Available tools: {[t.tool_name for t in tools]}")

        agent = Agent(
            model=BedrockModel(model_id=NOVA_PRO_MODEL_ID, max_tokens=4096),
            system_prompt="You are a financial research assistant. Use web search to find current information about financial regulations and markets.",
            tools=tools,
        )

        agent("Validate BSB 082-401 and search to verify that ABN 51824753556 belongs to Commonwealth Bank")
except Exception as e:
    if "404" not in str(e):
        raise e


Connecting to deployed MCP server...
Available tools: ['websearch', 'validate_bsb']
<thinking> First, I need to validate the BSB number 082-401 to identify the bank. Then, I will search to verify that the ABN 51824753556 belongs to Commonwealth Bank. </thinking>

Tool #1: validate_bsb
<thinking> The BSB 082-401 is valid and belongs to National Australia Bank. Now, I need to search to verify that the ABN 51824753556 belongs to Commonwealth Bank. </thinking> 
Tool #2: websearch
<thinking> The search results indicate that the ABN 51824753556 is associated with the Australian Taxation Office, not Commonwealth Bank. Therefore, the ABN 51824753556 does not belong to Commonwealth Bank. </thinking>

The ABN 51824753556 is associated with the Australian Taxation Office, not Commonwealth Bank. Therefore, the ABN 51824753556 does not belong to Commonwealth Bank.

Session termination failed: 404


## Examining the Agent Loop

The agent loop shows how the agent decided to use the web search tool:

In [10]:
from rich.table import Table
import rich
import json

console = rich.get_console()
console.print(f'Agent Loop Cycles: {agent.event_loop_metrics.cycle_count}')

table = Table(title='Agent Messages', show_lines=True)
table.add_column('Role', style='green', width=10)
table.add_column('Text', style='magenta', max_width=50)
table.add_column('Tool', style='cyan', width=20)
table.add_column('Input', style='cyan', max_width=30)
table.add_column('Result', style='cyan', max_width=30)

for msg in agent.messages[-6:]:
    text = [c['text'] for c in msg['content'] if 'text' in c]
    tool_name = [c['toolUse']['name'] for c in msg['content'] if 'toolUse' in c]
    tool_input = [c['toolUse']['input'] for c in msg['content'] if 'toolUse' in c]
    tool_result = [c['toolResult']['content'][0] for c in msg['content'] if 'toolResult' in c]
    table.add_row(
        msg['role'],
        (text[-1][:100] + '...') if text and len(text[-1]) > 100 else (text[-1] if text else ''),
        tool_name[-1] if tool_name else '',
        (json.dumps(tool_input[-1])[:80] + '...') if tool_input else '',
        (json.dumps(tool_result[-1])[:80] + '...') if tool_result else '',
    )

console.print(table)

Agent Loop Cycles: 2

                                                  Agent Messages                                                   
┏━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Role       ┃ Text                    ┃ Tool                 ┃ Input                   ┃ Result                  ┃
┡━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ user       │ Validate BSB 082-401    │                      │                         │                         │
│            │ and search to verify    │                      │                         │                         │
│            │ that ABN 51824753556    │                      │                         │                         │
│            │ belongs to Commonwealth │                      │                         │                         │
│            │ Bank                    │                      │                         │                         │
├────────────┼─────────────────────────┼──────────────────────┼─────────────────────────┼─────────────────────────┤
│ assistant  │ <thinking> To validate  │ websearch            │ {"keywords": "ABN       │                         │
│            │ the BSB 082-401, I will │                      │ 51824753556             │                         │
│            │ use the `validate_bsb`  │                      │ Commonwealth Bank",     │                         │
│            │ tool. Then, to verify   │                      │ "max_results": 5,       │                         │
│            │ that ABN...             │                      │ "region": "u...         │                         │
├────────────┼─────────────────────────┼──────────────────────┼─────────────────────────┼─────────────────────────┤
│ user       │                         │                      │                         │ {"text": "{\n           │
│            │                         │                      │                         │ \"title\": \"Australian │
│            │                         │                      │                         │ Tax Office transparency │
│            │                         │                      │                         │ and corporate ban...    │
├────────────┼─────────────────────────┼──────────────────────┼─────────────────────────┼─────────────────────────┤
│ assistant  │ <thinking> The BSB      │                      │                         │                         │
│            │ 082-401 is valid and    │                      │                         │                         │
│            │ belongs to the National │                      │                         │                         │
│            │ Australia Bank. The web │                      │                         │                         │
│            │ search resul...         │                      │                         │                         │
└────────────┴─────────────────────────┴──────────────────────┴─────────────────────────┴─────────────────────────┘

## Resource Cleanup (Optional)

In [ ]:
agentcore_runtime.delete()
cognito_client.delete_user_pool(UserPoolId=user_pool_id)
print('✅ Resources cleaned up')

## Summary

In this lab, you:

- ✅ Created a custom MCP server with web search functionality
- ✅ Set up Amazon Cognito for secure authentication
- ✅ Deployed the MCP server to Bedrock AgentCore Runtime
- ✅ Connected a Strands Agent to the deployed server with authentication
- ✅ Used the agent to search for FSI regulatory and market information

### Key Benefits of AgentCore Runtime for MCP

- **Scalable Deployment**: Managed infrastructure for MCP servers
- **Secure Authentication**: Built-in Cognito JWT verification
- **Easy Integration**: Seamless connection with Strands Agents
- **Production Ready**: Enterprise-grade reliability and monitoring